In [ ]:
from pathlib import Path

BIBLE_VERSES_PATH = Path("../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible-verses.csv")
DATABASE_PATH = Path("biblia.db")
MAKE_VECTOR_STORE = False
MAKE_DATABASE = True

In [ ]:
from pprint import pprint

import pandas as pd

from bible_model import Verse

REQUIRED_COLUMNS = {
    "book", "chapter", "verse_start", "verse_end", "text",
    "verse_acc", "pdf_page", "need_review", "raw_verse_marker",
    "parse_issue",
}

df_verse = pd.read_csv(
    BIBLE_VERSES_PATH,
    dtype={
        "book": "string", "chapter": "Int64", "verse_start": "Int64",
        "verse_end": "Int64", "text": "string", "verse_acc": "Int64",
        "pdf_page": "Int64", "need_review": "boolean",
        "raw_verse_marker": "string", "parse_issue": "string",
    },
)

missing_columns = REQUIRED_COLUMNS.difference(df_verse.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no DataFrame: {sorted(missing_columns)}")

def optional_int(value: object) -> int | None:
    return None if pd.isna(value) else int(value)

def optional_str(value: object) -> str | None:
    return None if pd.isna(value) else str(value)

def verse_from_row(row: pd.Series) -> Verse:
    return Verse(
        book=str(row.book),
        chapter=int(row.chapter),
        verse_start=optional_int(row.verse_start),
        verse_end=optional_int(row.verse_end),
        text=str(row.text),
        verse_acc=optional_int(row.verse_acc),
        pdf_page=int(row.pdf_page),
        need_review=bool(row.need_review),
        raw_verse_marker=str(row.raw_verse_marker),
        parse_issue=optional_str(row.parse_issue),
    )

verses = [verse_from_row(row) for _, row in df_verse.iterrows()]
pprint(verses[0])

print(f"{len(verses)} blocos de versículos carregados.")

In [ ]:
from typing import List
from langchain.schema import Document

def verse_metadata(verse: Verse) -> dict[str, str | int | bool]:
    return {
        name: value
        for name, value in vars(verse).items()
        if value is not None
    }

docs: List[Document] = [
    Document(page_content=verse.text, metadata=verse_metadata(verse))
    for verse in verses
]

print(docs[0])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

if MAKE_VECTOR_STORE:
    # 3. Inicializa o modelo de embedding
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    db = Chroma(
        collection_name="biblia",
        embedding_function=embedding_model,
        persist_directory="biblia_vectorstore",
    )
    # 4. Cria o índice vetorial
    ids = db.add_documents(docs)

In [ ]:
if MAKE_VECTOR_STORE:
    query = "No princípio, Deus criou o céu e a terra."
    results = db.similarity_search(query, k=5)

    for result in results:
        print(result)

In [ ]:
import sqlite3

CREATE_TABLE_SQL = """
CREATE TABLE versiculos (
    id INTEGER PRIMARY KEY,
    book TEXT NOT NULL,
    chapter INTEGER NOT NULL,
    verse_start INTEGER,
    verse_end INTEGER,
    text TEXT NOT NULL,
    verse_acc INTEGER,
    pdf_page INTEGER NOT NULL,
    need_review BOOLEAN NOT NULL,
    raw_verse_marker TEXT NOT NULL,
    parse_issue TEXT,
    CHECK (need_review IN (0, 1)),
    CHECK (verse_start IS NULL OR verse_end IS NULL OR verse_start <= verse_end)
)
"""

INSERT_VERSE_SQL = """
INSERT INTO versiculos (
    book, chapter, verse_start, verse_end, text, verse_acc, pdf_page,
    need_review, raw_verse_marker, parse_issue
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

if MAKE_DATABASE:
    rows = [
        (
            verse.book, verse.chapter, verse.verse_start, verse.verse_end,
            verse.text, verse.verse_acc, verse.pdf_page, int(verse.need_review),
            verse.raw_verse_marker, verse.parse_issue,
        )
        for verse in verses
    ]

    with sqlite3.connect(DATABASE_PATH) as conn:
        conn.execute("DROP TABLE IF EXISTS versiculos")
        conn.execute(CREATE_TABLE_SQL)
        conn.executemany(INSERT_VERSE_SQL, rows)
        conn.execute(
            "CREATE INDEX idx_versiculos_reference ON versiculos (book, chapter, verse_start)"
        )

    print(f"{len(rows)} blocos inseridos em {DATABASE_PATH}.")